##### AI Employee Approval Assistant
##### setup and prompting
1. openai python client.
2. chat completions api.
3. models.
4. system prompt.
5. user prompt.
6. parameters.
7. markdown responses

In [4]:
# !pip install openai python-dotenv

In [5]:
## import required libraries
from openai import OpenAI
from dotenv import load_dotenv
import os 
from IPython.display import Markdown, display

In [6]:
## load environment variables
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"SUCCESS: OPENAI_API_KEY is present.")
else:
    print(f"FAILURE: OPENAI_API_KEY is not present.")

SUCCESS: OPENAI_API_KEY is present.


In [7]:
## creating an openai client
client = OpenAI(api_key=api_key)

In [8]:
## choosing the model
MODEL = "gpt-4o-mini"

In [9]:
## defining the system prompt
SYSTEM_PROMPT = """
You are an AI Employee Expense Approval Assistant.

Responsibilities:
- Review employee expense Claim.
- Follow company expense policies.
- Identify missing information.
- Ask follow-up questions whenever required.
- Explain every decision clearly.
- Always respond in Markdown.

Formatting Rules:
- Use headings.
- Use bullet points.
- Use tables whenever appropriate.
- Highlight important information using bold text.
- Never return plain text.
""" 


In [10]:
## creating a reusable chat completion function
def ask_gpt(user_prompt, system_prompt=SYSTEM_PROMPT):

    # send the conversation to the model
    response = client.chat.completions.create(
        model=MODEL,
        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        # control response randomness
        temperature=0.3
    )

    # return only the generated response
    return response.choices[0].message.content

In [11]:
## first chat completion
## employee submits a response
user_prompt = """ 
Hotel expense: ₹12,000
"""

## ask GPT
reply = ask_gpt(user_prompt)

# print raw Markdown
print(reply)

# Expense Claim Review

## Submitted Expense
- **Type:** Hotel Expense
- **Amount:** ₹12,000

## Review of Expense

### Company Policy Check
- **Policy Compliance:** 
  - **Allowed Amount:** Please confirm if ₹12,000 is within the allowed limit for hotel expenses as per company policy.
  - **Purpose of Stay:** Was this stay for a business-related event or meeting?

### Missing Information
To proceed with the approval, I need the following information:
- **Dates of Stay:** Please provide the check-in and check-out dates.
- **Location:** Where was the hotel located?
- **Receipt:** Is there a receipt available for this expense?

## Next Steps
Please provide the requested information so I can assist you further with the approval process. Thank you!


In [12]:
## render markdown
display(Markdown(reply))


# Expense Claim Review

## Submitted Expense
- **Type:** Hotel Expense
- **Amount:** ₹12,000

## Review of Expense

### Company Policy Check
- **Policy Compliance:** 
  - **Allowed Amount:** Please confirm if ₹12,000 is within the allowed limit for hotel expenses as per company policy.
  - **Purpose of Stay:** Was this stay for a business-related event or meeting?

### Missing Information
To proceed with the approval, I need the following information:
- **Dates of Stay:** Please provide the check-in and check-out dates.
- **Location:** Where was the hotel located?
- **Receipt:** Is there a receipt available for this expense?

## Next Steps
Please provide the requested information so I can assist you further with the approval process. Thank you!

In [13]:
## Another example
user_prompt = """
Flight expense: ₹25,000

Purpose: Client Meeting

Location: London
"""

reply=ask_gpt(user_prompt)
display(Markdown(reply))

# Expense Claim Review

## Summary of the Claim
- **Expense Type:** Flight
- **Amount:** ₹25,000
- **Purpose:** Client Meeting
- **Location:** London

## Review Against Company Policies

### Policy Compliance Check
- **Purpose of Travel:** Client meetings are generally acceptable under company policy.
- **Location:** London is a valid business destination.
- **Expense Amount:** ₹25,000 for a flight needs to be justified based on the following:
  - **Class of Travel:** Economy, Business, or First Class?
  - **Booking Method:** Was it booked through the company’s preferred travel agency?
  - **Travel Dates:** Are the dates aligned with the meeting schedule?

## Missing Information
To proceed with the approval, I need the following details:

1. **Class of Travel:** What class was the flight booked in?
2. **Booking Method:** How was the flight booked?
3. **Travel Dates:** What were the departure and return dates?

## Next Steps
Please provide the missing information so I can finalize the review of your expense claim. Thank you!

In [14]:
# why do we use 2 prompts?
# system prompt: defines AI's behaviour.
# user prompt: defines today's request.
# GPT: Markdown the response

In [15]:
## parameters
response = client.chat.completions.create(
    model = MODEL,
    messages=[
        {
            "role":"system",
            "content": SYSTEM_PROMPT
        },
        {
            "role":"user",
            "content":"Review hotel expense $1400."
        }
    ],
    # lower values make responses more consistent
    temperature=0.2
)

## experiment with parameters
reply = ask_gpt(
    """
Review hotel expense of $1400."""
)
display(Markdown(reply))

# Hotel Expense Review

## Expense Details
- **Amount:** $1400
- **Type:** Hotel Accommodation

## Company Expense Policy Review
To ensure compliance with company policies, I will check the following criteria:

1. **Policy Limits:**
   - What is the maximum allowable amount for hotel accommodations per night?
  
2. **Duration of Stay:**
   - How many nights was the hotel booked for?
  
3. **Approval:**
   - Was prior approval obtained for this expense?

4. **Receipts:**
   - Is there a receipt attached to this claim?

## Missing Information
To proceed with the review, I need the following information:

- **Duration of Stay:** How many nights was the hotel booked for?
- **Approval Confirmation:** Was this expense pre-approved?
- **Receipt:** Is there a receipt available for this expense?

Please provide the above details to continue with the review. Thank you!

##### AI Employee Expense Approval Assistant
##### Section 2 - Smarter Assistant

##### concepts covered
1. tokenization (tiktoken)
2. Context Windows
4. Token Limits
5. API Costs
6. JSON Prompting.
7. Building JSON Prompts.
8. Chaining GPT Calls.

In [16]:
# Install the tokenizer library
!pip install tiktoken


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
## import tiktoken
import tiktoken

In [18]:
## create a tokenizer
## load the tokenizer for the selected model
encoding = tiktoken.encoding_for_model(MODEL)


In [19]:
## count the tokens
def count_tokens(text):
    # convert text into tokens
    tokens=encoding.encode(text)

    return len(tokens)

In [20]:
## example
expense = """
Hotel Expense $1200
Food: $200
Taxi: %300
""" 

print(count_tokens(expense))

17


Why tokenization matters
LLMs do not read text word-by-word
they first convert text into smaller units called tokens.

example: 
Expense reimbursement approved.
Expense, re, imbursement, approved.

Every prompt is converted into tokens before it reaches the model.

In [21]:
## context window
long_history = """ 
Employee Expenses

January...
February...
March...
April...
May...
June...
July...
August...
September...
October...
November...
December...
"""

In [22]:
## count context tokens
print(count_tokens(long_history))

28


Why context window matters?
the context window is the maximum number of tokens the model can process in one request.

the total includes: system prompt + user prompt + previous conversation + model repsonse.

if the total exceeds the model's context window, older information may need to be truncated or summarized.

In [23]:
# Return both the response and token usage
def ask_gpt(user_prompt, system_prompt=SYSTEM_PROMPT):

    # Send the request to the model
    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        temperature=0.3

    )

    # Extract the assistant's reply
    reply = response.choices[0].message.content

    # Extract usage statistics
    usage = response.usage

    # Return both
    return reply, usage

In [24]:
# use the updated function
reply, usage = ask_gpt(
    """ Review this expense.
    Hotel: $1400."""
)

display(Markdown(reply))

# Expense Review: Hotel Claim

## Summary of Claim
- **Expense Type:** Hotel
- **Claim Amount:** $1400

## Company Expense Policy
Before approving the expense, let's review the company's expense policy regarding hotel stays:

- **Maximum Allowable Rate:** $250 per night (if applicable)
- **Justification Required:** Purpose of stay (business meeting, conference, etc.)
- **Duration of Stay:** Number of nights must be specified
- **Location:** Must be within the approved travel area

## Missing Information
To proceed with the review, I need the following information:

1. **Duration of Stay:**
   - How many nights was the hotel booked for?

2. **Purpose of Stay:**
   - What was the reason for the hotel stay? (e.g., business meeting, conference)

3. **Location:**
   - Where was the hotel located? (City/Area)

## Next Steps
Please provide the missing information so I can complete the review of this expense claim. Thank you!

In [25]:
## view token usage
# prompt tokens
print("Prompt tokens: ", usage.prompt_tokens)
# completion tokens
print("Completion tokens: ", usage.completion_tokens)
# total tokens
print("Total tokens: ", usage.total_tokens)

Prompt tokens:  103
Completion tokens:  210
Total tokens:  313


In [26]:
# Estimate the API Cost
# Example pricing - replace with the current pricing 
# for your choosen model
INPUT_COST_PER_MILLION = 1.25
OUTPUT_COST_PER_MILLION = 10.00

In [27]:
## estimate the request cost
def estimate_cost(usage):
    # calculate input cost
    input_cost = (
        usage.prompt_tokens / 1_000_000
    ) * INPUT_COST_PER_MILLION

    # calculate output cpst
    output_cost = (
        usage.completion_tokens / 1_000_000
    ) * OUTPUT_COST_PER_MILLION

    # return total cost
    return input_cost + output_cost

In [28]:
print(f"${estimate_cost(usage):.8f}")

$0.00222875


In [29]:
# Token limits: 
# Every model has a maximum context window.

# prompt tokens + completion tokens <= Maximum Context Window
# If this limit is exceeded, the request will fail or require truncation.

In [30]:
## json prompting
json_prompt = """ 
Extract the expense information

Return ONLY VALID JSON
{
    "employee":"",
    "department":"",
    "expenses":[]
}
Expense:

Employee: John
Department: Sales
Hotel ₹15000
Food ₹2500
Taxi ₹800
"""

In [31]:
# GPT generates JSON
reply, usage = ask_gpt(json_prompt)
print(reply)

```json
{
    "employee": "John",
    "department": "Sales",
    "expenses": [
        {
            "category": "Hotel",
            "amount": 15000
        },
        {
            "category": "Food",
            "amount": 2500
        },
        {
            "category": "Taxi",
            "amount": 800
        }
    ]
}
```


Why JSON?
1. Easy for humans to read.
JSON: Easy for Python Programmers to Process.

Chaining GPT calls:
Employee writes an expense -> GPT Call 1 (extract structured JSON) -> Python validates the JSON. -> GPT Call 2 -> Generate a Professional Markdown report from validated JSON.

In [32]:
# GPT Call 1
json_result, usage = ask_gpt(json_prompt)
print(json_result)

## GPT call 2
markdown_prompt = f""" 
Create a Professional expense review report.

Expense JSON
{json_result}

Return Markdown only.
"""

report, usage = ask_gpt(markdown_prompt)
display(Markdown(report))

```json
{
    "employee": "John",
    "department": "Sales",
    "expenses": [
        {
            "type": "Hotel",
            "amount": 15000
        },
        {
            "type": "Food",
            "amount": 2500
        },
        {
            "type": "Taxi",
            "amount": 800
        }
    ]
}
```


# Employee Expense Review Report

## Employee Information
- **Name:** John
- **Department:** Sales

## Expense Summary
| **Expense Type** | **Amount (in currency units)** |
|-------------------|-------------------------------|
| Hotel             | 15,000                        |
| Food              | 2,500                         |
| Taxi              | 800                           |
| **Total**        | **18,300**                   |

## Expense Review

### 1. Hotel Expense
- **Amount:** 15,000
- **Review:** 
  - Please confirm if this hotel expense is within the company policy limit for accommodation.
  
### 2. Food Expense
- **Amount:** 2,500
- **Review:** 
  - Ensure that this food expense complies with the daily meal allowance set by the company policy.

### 3. Taxi Expense
- **Amount:** 800
- **Review:** 
  - Verify if receipts are available for this taxi expense.

## Missing Information
- **Receipts:** Please provide receipts for all expenses, especially the taxi expense.
- **Policy Confirmation:** Confirm if the hotel and food expenses align with company policy limits.

## Next Steps
- **Action Required:** 
  - Please provide the missing receipts and confirm compliance with the company policy for hotel and food expenses.
  
Thank you for your cooperation!

Why Chaining GPT Calls?
One large prompt often tries to do everything at once.

Instead, break the task into smaller, focused steps:

1. Extract data.
2. Validate data.
3. Generate the final report.

This approach improves readability, maintainability, and makes debugging easier.

AI Employee Approval Assistant 
Section 3-Building the AI Workflow

1. OpenAI tool calling.
2. Python Functions (Tools).
3. JSON response Format.
4. Tool Execution.
5. Chaining GPT Calls.
6. Expense Validation.
7. Policy Checking.
8. Final Markdown Report.

In [33]:
"""Company Expense Policy"""

# define the company expense policy
EXPENSE_POLICY = {
    "Hotel": 15000,
    "Food": 3000,
    "Taxi": 1000,
    "Flight": 40000
}

In [34]:
# create a python tool
# calcuate the total expense amount
def calculate_total(expenses):
    # start with 0
    total = 0 
    # add each expense amount
    for expense in expenses:
        total += expense["amount"]
    
    return total

In [35]:
# tool to check whether each expense follows company policy
def check_policy(expenses):
    # store policy violations
    violations = []
    # check every expense
    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]

        # skip categories that are not in the policy
        if category not in EXPENSE_POLICY:
            continue
        # compare against the policy limit
        if amount > EXPENSE_POLICY[category]:
            violations.append({
                "category": category,
                "amount": amount,
                "limit": EXPENSE_POLICY[category]
            })

    
    # return all violations
    return violations

In [36]:
# Describe the available tools to the model
TOOLS = [

    {

        "type": "function",

        "function": {

            "name": "calculate_total",

            "description": "Calculate the total expense amount.",

            "parameters": {

                "type": "object",

                "properties": {

                    "expenses": {

                        "type": "array",

                        "items": {

                            "type": "object",

                            "properties": {

                                "category": {

                                    "type": "string"

                                },

                                "amount": {

                                    "type": "number"

                                }

                            }

                        }

                    }

                },

                "required": [

                    "expenses"

                ]

            }

        }

    },

    {

        "type": "function",

        "function": {

            "name": "check_policy",

            "description": "Check company expense policy.",

            "parameters": {

                "type": "object",

                "properties": {

                    "expenses": {

                        "type": "array",

                        "items": {

                            "type": "object",

                            "properties": {

                                "category": {

                                    "type": "string"
                                },
                                "amount": {
                                   "type": "number"

                                }
                            }
                        }
                    }
                },

                "required": [
                    "expenses"

                ]
            }
        }
    }
]

In [37]:
## employee expenses
# employee submits an expense request
expense_request = """ 
Employee: John
Department: Sales
Flight ₹38000
Hotel ₹18000
Food ₹2500
Taxi ₹800
"""

In [38]:
## extract JSON properly
# ask the model to return structured JSON
response = client.chat.completions.create(
    model=MODEL,
    response_format={
        "type": "json_object"
    },
    messages=[
        {
            "role":"system",
            "content":"Extract the expense details into JSON."
        },
        {
            "role":"user",
            "content": expense_request
        }
    ]
)

expense_json = response.choices[0].message.content 
print(expense_json)

{
  "employee": {
    "name": "John",
    "department": "Sales"
  },
  "expenses": {
    "flight": 38000,
    "hotel": 18000,
    "food": 2500,
    "taxi": 800
  }
}


In [39]:
## convert json string into python dictionary
import json 
# convert JSON text into a python dictionary
expense_data = json.loads(expense_json)

In [40]:
## execute python tools
total = calculate_total(
    expense_data['expenses']
)

# check company policy 
violations = check_policy(
    expense_data["expenses"]
)

TypeError: string indices must be integers, not 'str'

In [ ]:
print(total)

print()

print(violations)

NameError: name 'total' is not defined

In [ ]:
# ## why tool calling? 
# GPT calculates numbers itself,
# sometimes inaccurate 
# with tool calling -> gpt calls python tool
# python performs the calculation and gpt explains the result 
# much more reliable.

In [42]:
## cell 13 openai tool calling automatic
# let the model decide when to call tools

response = client.chat.completions.create(
    model=MODEL,
    messages = [
        {
            "role":"user",
            "content": expense_request
        }
    ],
    tools = TOOLS,
    tool_choice = "auto"
)


In [43]:
# cell 14 - view tool calls

tool_calls = response.choices[0].message.tool_calls
print(tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_P2fFI9gL1Koo4lzbM5wadO5r', function=Function(arguments='{"expenses": [{"category": "Flight", "amount": 38000}, {"category": "Hotel", "amount": 18000}, {"category": "Food", "amount": 2500}, {"category": "Taxi", "amount": 800}]}', name='calculate_total'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_j72jCCeQqslKHi3BlUsHPtJ7', function=Function(arguments='{"expenses": [{"category": "Flight", "amount": 38000}, {"category": "Hotel", "amount": 18000}, {"category": "Food", "amount": 2500}, {"category": "Taxi", "amount": 800}]}', name='check_policy'), type='function')]


In [44]:
## execute requested tools
## execute each requested tool

for tool_call in tool_calls:
    # read the tool_name
    tool_name = tool_call.function.name 
    # read the tool arguments
    arguments = json.loads(tool_call.function.arguments)

    ## execute calcuate_total
    if tool_name == "calculate_total":
        result = calculate_total(arguments["expenses"])
    ## execute check policy
    elif tool_name == "check_policy":
        result = check_policy(
            arguments["expenses"]
        )
    # display the result
    print(tool_name)
    print(result)
    print()

calculate_total
59300

check_policy
[{'category': 'Hotel', 'amount': 18000, 'limit': 15000}]



In [45]:
expense_request = """

Employee : Sarah

Department : Finance

Flight ₹45000

Hotel ₹12000

Food ₹1800

Taxi ₹950

"""

json_result = client.chat.completions.create(

    model=MODEL,

    response_format={

        "type":"json_object"

    },

    messages=[

        {

            "role":"system",

            "content":"Extract expense information."

        },

        {

            "role":"user",

            "content":expense_request

        }

    ]

)

expense_data = json.loads(

    json_result.choices[0].message.content

)

total = calculate_total(

    expense_data["expenses"]

)

violations = check_policy(

    expense_data["expenses"]

)

report_prompt = f"""

Employee

{expense_data["employee"]}

Department

{expense_data["department"]}

Expenses

{expense_data["expenses"]}

Total

₹{total}

Violations

{violations}

Generate a Markdown report.

"""

report = ask_gpt(

    report_prompt

)[0]

display(Markdown(report))

BadRequestError: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

for example:
- Employee submits expense.
- Chat completions API
- Tokenizer
- Tokens
- Transformer
- Attention Mechanism
- LLM
- Reasoning
- Tool Calling
- Python Functions
- Markdown report

1. openai Python client - creating the client
2. chat completions api - every model call
3. system prompt - defining the AIs role
4. user prompt - employee expense submission
5. models - MODEL = "gpt-4o-mini"
6. parameters - temperature, etc.
7. Tokenization - Couting Prompt tokens
8. Context Window - Long expense history
9. API costs - cost estimation history
10. JSON prompting - Expense extraction
11. tool calling - policy checking and total calculation
12. chaining GPT calls - Extract -> validate -> Report
13. Transformers - How the model processes the tokens
14. Attention - How relevant expense details are identified.
15. Emergent Intelligence - The model's ability to infer missing details or suggest follow-up questions.
16. Agentic AI - coordinating multiple GPT calls and python tools to complete a task
17. Base vs Chat vs Reasoning - Choosing the right model for different stages.
18. Deep research - extending the assistant to consult company policy documents or external references before making a recommendation.